# Lab 10 - Apple Brand Monitor Data Analysis

This notebook adapts the Lab 10 movie analytics workflow to the `social_media_brand_monitor` project. The cleaned dataset is restricted to Apple mentions, then combined with MySQL and MongoDB results for reshaping, aggregation, pivoting, time-series analysis, and analytical reporting.

## Project-Specific Fields

This submission is for the `social_media_brand_monitor` project focused on Apple mentions, not a movie dataset.

- cleaned input file: `data/processed/cleaned/cleaned_data.csv`
- MySQL table: `apple_article_metrics`
- numeric analysis fields: `rating`, `title_length`, `overview_length`
- grouping fields: `mention_year`, `primary_keyword`
- date field: `mention_date`

In [1]:
from pathlib import Path
import logging
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.analytics.aggregator import add_length_metrics, source_summary, top_n_per_group, yearly_trends
from src.analytics.data_combiner import compare_join_types, merge_on_key, save_join_comparison_chart
from src.analytics.db_connector import create_article_metrics_table, get_mysql_connection, populate_article_metrics, query_article_metrics
from src.analytics.explorer import filter_apple_mentions
from src.analytics.insight_reporter import (
    language_distribution,
    run_all_questions,
    save_insight_summary,
    save_keyword_chart,
    save_language_distribution_chart,
    save_question_report,
    save_source_share_chart,
    save_yearly_volume_chart,
    source_share_summary,
    top_keywords_by_mention_count,
    yearly_release_volume,
)
from src.analytics.mongo_pipeline import build_source_mentions_pipeline, run_pipeline as run_mongo_aggregation
from src.analytics.pivot_builder import add_primary_keyword, build_keyword_year_pivot, build_language_year_crosstab, melt_metrics
from src.analytics.time_series import add_rolling_averages, build_monthly_mentions, parse_mention_dates, resample_mentions, save_rolling_mentions_chart
from src.utils.logger import get_logger

logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")
logger = get_logger("lab10_data_analysis")

CLEANED_CSV_PATH = PROJECT_ROOT / "data" / "processed" / "cleaned" / "cleaned_data.csv"
ANALYTICS_DIR = PROJECT_ROOT / "data" / "processed" / "analytics"
ANALYTICS_DIR.mkdir(parents=True, exist_ok=True)


## Load and Prepare the Cleaned Apple Dataset

In [2]:
cleaned_df = pd.read_csv(CLEANED_CSV_PATH)
cleaned_df = filter_apple_mentions(cleaned_df, keyword="apple")
cleaned_df = parse_mention_dates(cleaned_df)
cleaned_df["rating"] = pd.to_numeric(cleaned_df.get("rating"), errors="coerce")
cleaned_df = add_length_metrics(cleaned_df)
cleaned_df = add_primary_keyword(cleaned_df)
logger.info("Loaded and prepared %s Apple rows for Lab 10 analysis", len(cleaned_df))
cleaned_df.head()

INFO: Apple mention filter complete | keyword=apple | rows_before=101 | rows_after=101
INFO: Parsed mention dates | valid_dates=101 | missing_dates=0
INFO: Primary keyword column created | unique_keywords=8
INFO: Loaded and prepared 101 Apple rows for Lab 10 analysis


,_id,source,author,content,description,document_type,extraction_timestamp,publishedAt,title,url,...,record_date,overview,mention_date,mention_year,mention_month,mention_weekday,mention_quarter,title_length,overview_length,primary_keyword
0,69ce9d979d3427d3654d3276,newsapi_Apple_page_2.json,Jusuf Hatic,Tim Cook hatte so einige Produkte in seiner Ze...,Nach über 15 Jahren hört Tim Cook als Apple-CE...,json,2026-04-25 20:39:13.673000+00:00,2026-04-24 19:59:00+00:00,News: Nach Rücktritt - »Mein erster großer Feh...,"https://www.gamestar.de/artikel/,3452032.html",...,NaN,Nach über 15 Jahren hört Tim Cook als Apple-CE...,2026-04-24 00:00:00+00:00,2026,4,Friday,2,103,190,tim cook
1,69ce9d979d3427d3654d3278,sample.csv,Alex Brown,Unknown,Unknown,csv,2026-04-25 20:39:14.090000+00:00,NaN,Apple doesnt lawsuit,NaN,...,NaN,Unknown,2026-03-02 00:00:00+00:00,2026,3,Monday,1,20,7,general_apple
2,69ce9d979d3427d3654d3279,sample.xml,Alex Brown,Unknown,Unknown,xml,2026-04-25 20:39:14.100000+00:00,NaN,Apple faces lawsuit,NaN,...,NaN,Unknown,2026-03-03 00:00:00+00:00,2026,3,Tuesday,1,19,7,general_apple
3,69ed2b5ef527cb7a1c1159e2,newsapi_Apple_page_1.json,Juli Clover,Apple is planning to start showing ads in the ...,Apple is planning to start showing ads in the ...,json,2026-04-25 21:11:28.488000+00:00,2026-04-24 20:58:00+00:00,Ads Are Coming to Apple Maps This Summer: Here...,https://www.macrumors.com/2026/04/24/apple-map...,...,NaN,Apple is planning to start showing ads in the ...,2026-04-24 00:00:00+00:00,2026,4,Friday,2,63,252,general_apple
4,69ed2b5ef527cb7a1c1159eb,newsapi_Apple_page_1.json,sara.schwartz@expressen.se (Sara Schwartz Wiman),Den ideella föreningen Biosfärområde på Österl...,Techjätten Apple verkar inte vara så sugna på ...,json,2026-04-25 21:02:33.444000+00:00,2026-04-24 20:42:58+00:00,Apple ville sätta stopp för föreningslogga i Ö...,https://www.expressen.se/nyheter/sverige/apple...,...,NaN,Techjätten Apple verkar inte vara så sugna på ...,2026-04-24 00:00:00+00:00,2026,4,Friday,2,53,196,general_apple


## MySQL Connection and Relational Table

This section fulfills the relational-database requirement: connect to MySQL, create the Apple metrics table, populate it from the cleaned CSV-derived DataFrame, and read it back with `pd.read_sql()`.

In [3]:
mysql_connection = get_mysql_connection()
create_article_metrics_table(mysql_connection)
populate_article_metrics(mysql_connection, cleaned_df)
mysql_df = query_article_metrics(where_clause="mention_year IS NOT NULL")
mysql_connection.close()
mysql_df.head()

INFO: Opened MySQL connection | host=localhost | port=3306 | database=social_media_brand_monitor
INFO: Ensured MySQL table exists: apple_article_metrics
INFO: Populated MySQL article metrics | rows_processed=101 | table=apple_article_metrics
INFO: Queried MySQL article metrics | rows=101 | table=apple_article_metrics


,mention_id,title,source_name,source_domain,author_name,document_type,content_type,language_code,mention_year,rating_value,title_length,overview_length,published_at,mention_date
0,69ce9d979d3427d3654d3276,News: Nach Rücktritt - »Mein erster großer Feh...,newsapi_Apple_page_2.json,www.gamestar.de,Jusuf Hatic,json,Unknown,NaN,2026,3.0,103,190,2026-04-24 19:59:00,2026-04-24
1,69ce9d979d3427d3654d3278,Apple doesnt lawsuit,sample.csv,NaN,Alex Brown,csv,Unknown,NaN,2026,3.0,20,7,NaT,2026-03-02
2,69ce9d979d3427d3654d3279,Apple faces lawsuit,sample.xml,NaN,Alex Brown,xml,Unknown,NaN,2026,3.0,19,7,NaT,2026-03-03
3,69ed2b5ef527cb7a1c1159e2,Ads Are Coming to Apple Maps This Summer: Here...,newsapi_Apple_page_1.json,www.macrumors.com,Juli Clover,json,Unknown,NaN,2026,3.0,63,252,2026-04-24 20:58:00,2026-04-24
4,69ed2b5ef527cb7a1c1159eb,Apple ville sätta stopp för föreningslogga i Ö...,newsapi_Apple_page_1.json,www.expressen.se,sara.schwartz@expressen.se (Sara Schwartz Wiman),json,Unknown,NaN,2026,3.0,53,196,2026-04-24 20:42:58,2026-04-24


## Combining Data from MySQL and the Cleaned CSV

In [4]:
metadata_df = cleaned_df[["_id", "title", "source", "document_type", "language", "mention_year", "mention_month", "mention_weekday", "mention_quarter", "overview", "mention_date", "rating", "title_length", "overview_length", "primary_keyword"]].copy()
mysql_join_df = mysql_df.rename(columns={"mention_id": "_id"}).copy()
mysql_join_df = mysql_join_df.rename(columns={column: f"mysql_{column}" for column in mysql_join_df.columns if column != "_id"})
join_comparison = compare_join_types(metadata_df, mysql_join_df, key="_id")
save_join_comparison_chart(join_comparison, ANALYTICS_DIR / "join_type_comparison.png")
join_comparison

INFO: Merged dataframes | how=inner | key=_id | left_rows=101 | right_rows=101 | merged_rows=101
INFO: Merged dataframes | how=left | key=_id | left_rows=101 | right_rows=101 | merged_rows=101
INFO: Merged dataframes | how=right | key=_id | left_rows=101 | right_rows=101 | merged_rows=101
INFO: Merged dataframes | how=outer | key=_id | left_rows=101 | right_rows=101 | merged_rows=101
INFO: Join comparison created | rows=4
INFO: Saved join comparison chart | path=c:\Users\topic\OneDrive\Desktop\PROJECTS\Projekat za unstructured data\social_media_brand_monitor\data\processed\analytics\join_type_comparison.png


,join_type,row_count
0,inner,101
1,left,101
2,right,101
3,outer,101


**Most appropriate join:** `left` join.

The cleaned Apple dataset is the primary analytical dataset, so every cleaned Apple mention should be preserved even if a relational enrichment row is missing. `left` join keeps the full cleaned dataset and adds MySQL metrics when available. `inner` could silently drop cleaned mentions, while `right` and `outer` are less appropriate because MySQL is a secondary enrichment source here.

In [5]:
combined_df = merge_on_key(metadata_df, mysql_join_df, key="_id", how="left")
combined_df.head()

INFO: Merged dataframes | how=left | key=_id | left_rows=101 | right_rows=101 | merged_rows=101


,_id,title,source,document_type,language,mention_year,mention_month,mention_weekday,mention_quarter,overview,...,mysql_author_name,mysql_document_type,mysql_content_type,mysql_language_code,mysql_mention_year,mysql_rating_value,mysql_title_length,mysql_overview_length,mysql_published_at,mysql_mention_date
0,69ce9d979d3427d3654d3276,News: Nach Rücktritt - »Mein erster großer Feh...,newsapi_Apple_page_2.json,json,NaN,2026,4,Friday,2,Nach über 15 Jahren hört Tim Cook als Apple-CE...,...,Jusuf Hatic,json,Unknown,NaN,2026,3.0,103,190,2026-04-24 19:59:00,2026-04-24
1,69ce9d979d3427d3654d3278,Apple doesnt lawsuit,sample.csv,csv,NaN,2026,3,Monday,1,Unknown,...,Alex Brown,csv,Unknown,NaN,2026,3.0,20,7,NaT,2026-03-02
2,69ce9d979d3427d3654d3279,Apple faces lawsuit,sample.xml,xml,NaN,2026,3,Tuesday,1,Unknown,...,Alex Brown,xml,Unknown,NaN,2026,3.0,19,7,NaT,2026-03-03
3,69ed2b5ef527cb7a1c1159e2,Ads Are Coming to Apple Maps This Summer: Here...,newsapi_Apple_page_1.json,json,NaN,2026,4,Friday,2,Apple is planning to start showing ads in the ...,...,Juli Clover,json,Unknown,NaN,2026,3.0,63,252,2026-04-24 20:58:00,2026-04-24
4,69ed2b5ef527cb7a1c1159eb,Apple ville sätta stopp för föreningslogga i Ö...,newsapi_Apple_page_1.json,json,NaN,2026,4,Friday,2,Techjätten Apple verkar inte vara så sugna på ...,...,sara.schwartz@expressen.se (Sara Schwartz Wiman),json,Unknown,NaN,2026,3.0,53,196,2026-04-24 20:42:58,2026-04-24


## Reshaping Data with `melt()` and `pivot_table()`

In [6]:
long_metrics = melt_metrics(
    combined_df,
    id_vars=["_id", "title", "source", "primary_keyword", "mention_year"],
    value_vars=["rating", "title_length", "overview_length"],
)
long_metrics.to_csv(ANALYTICS_DIR / "long_metrics.csv", index=False)
long_metrics.head()

INFO: Melted dataframe | rows=303 | metrics=['rating', 'title_length', 'overview_length']


,_id,title,source,primary_keyword,mention_year,metric_name,metric_value
0,69ce9d979d3427d3654d3276,News: Nach Rücktritt - »Mein erster großer Feh...,newsapi_Apple_page_2.json,tim cook,2026,rating,3.0
1,69ce9d979d3427d3654d3278,Apple doesnt lawsuit,sample.csv,general_apple,2026,rating,3.0
2,69ce9d979d3427d3654d3279,Apple faces lawsuit,sample.xml,general_apple,2026,rating,3.0
3,69ed2b5ef527cb7a1c1159e2,Ads Are Coming to Apple Maps This Summer: Here...,newsapi_Apple_page_1.json,general_apple,2026,rating,3.0
4,69ed2b5ef527cb7a1c1159eb,Apple ville sätta stopp för föreningslogga i Ö...,newsapi_Apple_page_1.json,general_apple,2026,rating,3.0


In [7]:
keyword_year_pivot = build_keyword_year_pivot(combined_df)
keyword_year_pivot.to_csv(ANALYTICS_DIR / "pivot_keyword_year.csv")
keyword_year_pivot

INFO: Keyword/year pivot table created | rows=2 | columns=9


primary_keyword,airpods,apple services,apple watch,general_apple,ipad,iphone,macbook,tim cook,All
mention_year,,,,,,,,,
2026,4,8,2,64,2,15,3,3,101
All,4,8,2,64,2,15,3,3,101


The pivot table above is the Apple-domain equivalent of a movie `release_year x primary_genre` summary. Here it shows Apple mention counts broken down by `mention_year` and `primary_keyword`, with `margins=True` to include subtotals.

In [8]:
language_year_crosstab = build_language_year_crosstab(combined_df)
language_year_crosstab.to_csv(ANALYTICS_DIR / "language_year_crosstab.csv")
language_year_crosstab

INFO: Language/year crosstab created | rows=3 | columns=2


mention_year,2026,All
language,,
de,7,7
en,7,7
All,14,14


## GroupBy Analysis

In [9]:
source_analysis = source_summary(combined_df)
source_analysis.to_csv(ANALYTICS_DIR / "source_analysis.csv", index=False)
source_analysis.head(10)

INFO: Source summary created | rows=6


,source,mention_count,avg_rating,median_rating,total_title_chars,avg_overview_chars
0,newsapi_Apple_page_1.json,39,3.0,3.0,2723,216.153846
1,newsapi_Apple_page_3.json,25,3.0,3.0,1816,215.36
2,newsapi_Apple_page_2.json,19,3.0,3.0,1572,220.473684
3,apple_ratings_large.csv,14,3.0,3.0,301,7.0
4,sample.csv,3,3.0,3.0,62,7.0
5,sample.xml,1,3.0,3.0,19,7.0


The grouped source summary uses named aggregations and multiple functions: `count`, `mean`, `median`, and `sum`.

In [10]:
yearly_analysis = yearly_trends(combined_df)
yearly_analysis.to_csv(ANALYTICS_DIR / "yearly_trends.csv", index=False)
yearly_analysis

INFO: Yearly trends created | rows=1


,mention_year,mention_count,unique_sources,avg_rating,avg_overview_chars
0,2026,101,6,3.0,179.49505


In [11]:
top_titles_by_keyword = top_n_per_group(combined_df, group_column="primary_keyword", sort_column="overview_length", n=3)
top_titles_by_keyword.to_csv(ANALYTICS_DIR / "top_titles_by_keyword.csv", index=False)
top_titles_by_keyword.head(15)

INFO: Top-N per group created | group_column=primary_keyword | sort_column=overview_length | n=3 | rows=22


,_id,title,source,document_type,language,mention_year,mention_month,mention_weekday,mention_quarter,overview,...,mysql_author_name,mysql_document_type,mysql_content_type,mysql_language_code,mysql_mention_year,mysql_rating_value,mysql_title_length,mysql_overview_length,mysql_published_at,mysql_mention_date
0,69ed2b5ef527cb7a1c115a14,"Deals: AirPods 4 for $99, $150 Off M5 MacBook ...",newsapi_Apple_page_3.json,json,NaN,2026,4,Friday,2,AirPods 4 are available at 22% off Apple’s ret...,...,OSXDaily,json,Unknown,NaN,2026,3.0,90,260,2026-04-24 19:56:44,2026-04-24
1,69fc8502ef3b485ee4bc2b4a,AirPods Max 2: Firmware-Update 8E258 ist da,newsapi_Apple_page_2.json,json,NaN,2026,5,Wednesday,2,Apple hat in der Nacht zu heute ein Firmware-U...,...,Admin,json,Unknown,NaN,2026,3.0,43,260,2026-05-06 12:11:18,2026-05-06
2,69fc8502ef3b485ee4bc2b50,Best Buy Mother's Day Deals: Up to 50% off + f...,newsapi_Apple_page_2.json,json,NaN,2026,5,Wednesday,2,Shop Mother's Day deals including up to $100 o...,...,Unknown,json,Unknown,NaN,2026,3.0,58,220,2026-05-06 12:09:54,2026-05-06
3,69ed2b5ef527cb7a1c1159fd,Jennifer Aniston Gets to Work Filming 'The Mor...,newsapi_Apple_page_2.json,json,NaN,2026,4,Friday,2,Jennifer Aniston is getting to work on the new...,...,Just Jared,json,Unknown,NaN,2026,3.0,65,260,2026-04-24 20:11:32,2026-04-24
4,69ed2b5ef527cb7a1c115a10,Schmigadoon!-dalok,newsapi_Apple_page_3.json,json,NaN,2026,4,Friday,2,Szerettem az Apple Tv+ komikus musical sorozat...,...,winnie,json,Unknown,NaN,2026,3.0,18,260,2026-04-24 20:00:00,2026-04-24
5,69f79537c7d32a5f5b914579,Coming Soon: What’s Next on Apple TV and Apple...,newsapi_Apple_page_1.json,json,NaN,2026,5,Saturday,2,"It’s a new month and it’s F1 season, which mea...",...,John Voorhees,json,Unknown,NaN,2026,3.0,65,260,2026-05-02 17:42:32,2026-05-02
6,69ed2b5ef527cb7a1c1159f9,How to watch Major League Baseball games Frida...,newsapi_Apple_page_2.json,json,NaN,2026,4,Friday,2,Friday Night Baseball is back on Apple TV toni...,...,Ryan Christoffel,json,Unknown,NaN,2026,3.0,60,205,2026-04-24 20:20:08,2026-04-24
7,69fc8502ef3b485ee4bc2b6a,The Weirdest Wearables From 100 Years Ago,newsapi_Apple_page_3.json,json,NaN,2026,5,Wednesday,2,The shockwatch of 1927 walked so Apple Watch c...,...,Matt Novak,json,Unknown,NaN,2026,3.0,41,55,2026-05-06 12:00:08,2026-05-06
8,69f79537c7d32a5f5b914566,Motorola Beats Apple And Samsung: The Razr Ult...,newsapi_Apple_page_1.json,json,NaN,2026,5,Saturday,2,Motorola Beats Apple And Samsung: The Razr Ult...,...,feedfeeder,json,Unknown,NaN,2026,3.0,93,260,2026-05-02 18:32:30,2026-05-02
9,69f79537c7d32a5f5b914572,Koniec z wadliwym ekranem. Najmniejszy tablet ...,newsapi_Apple_page_1.json,json,NaN,2026,5,Saturday,2,Od miesięcy mówiło się o wielkich premierach A...,...,Agnieszka Serafinowicz,json,Unknown,NaN,2026,3.0,90,260,2026-05-02 18:00:54,2026-05-02


## Time Series Analysis

In [12]:
print("Valid mention dates:", combined_df["mention_date"].notna().sum())
print("Missing mention dates:", combined_df["mention_date"].isna().sum())
combined_df[["mention_date", "mention_year", "mention_month", "mention_weekday", "mention_quarter"]].head()

Valid mention dates: 101
Missing mention dates: 0


,mention_date,mention_year,mention_month,mention_weekday,mention_quarter
0,2026-04-24 00:00:00+00:00,2026,4,Friday,2
1,2026-03-02 00:00:00+00:00,2026,3,Monday,1
2,2026-03-03 00:00:00+00:00,2026,3,Tuesday,1
3,2026-04-24 00:00:00+00:00,2026,4,Friday,2
4,2026-04-24 00:00:00+00:00,2026,4,Friday,2


In [13]:
monthly_mentions = build_monthly_mentions(combined_df)
monthly_mentions = add_rolling_averages(monthly_mentions)
monthly_mentions.to_csv(ANALYTICS_DIR / "monthly_mentions.csv", index=False)
save_rolling_mentions_chart(monthly_mentions, ANALYTICS_DIR / "rolling_mentions.png")
monthly_mentions.head()

INFO: Monthly mention series created | rows=3
INFO: Added rolling averages to monthly series | value_column=mention_count
INFO: Saved rolling mentions chart | path=c:\Users\topic\OneDrive\Desktop\PROJECTS\Projekat za unstructured data\social_media_brand_monitor\data\processed\analytics\rolling_mentions.png


,mention_date,mention_count,avg_rating,avg_overview_chars,mention_count_rolling_3,mention_count_rolling_6,mention_count_rolling_12
0,2026-03-31 00:00:00+00:00,4,3.0,7.0,4.000000,4.000000,4.000000
1,2026-04-30 00:00:00+00:00,35,3.0,127.971429,19.500000,19.500000,19.500000
2,2026-05-31 00:00:00+00:00,62,3.0,219.709677,33.666667,33.666667,33.666667


In [14]:
yearly_resampled = resample_mentions(combined_df, frequency="YE")
yearly_resampled.to_csv(ANALYTICS_DIR / "resampled_yearly_mentions.csv", index=False)
yearly_resampled

INFO: Resampled mention series created | frequency=YE | rows=1


,mention_date,mention_count,avg_rating,avg_overview_chars
0,2026-12-31 00:00:00+00:00,101,3.0,179.49505


## MongoDB Aggregation Pipeline

In [15]:
mongo_pipeline_df = run_mongo_aggregation(build_source_mentions_pipeline())
mongo_pipeline_df.to_csv(ANALYTICS_DIR / "mongo_source_mentions.csv", index=False)
mongo_pipeline_df.head(10)

INFO: MongoDB aggregation pipeline executed | rows=7


,mention_count,avg_rating,source
0,117,None,raw_brand_mentions.csv
1,42,None,newsapi_Apple_page_3.json
2,40,None,newsapi_Apple_page_1.json
3,36,None,newsapi_Apple_page_2.json
4,14,None,apple_ratings_large.csv
5,5,None,sample.csv
6,4,None,sample.xml


This pipeline uses `$match`, `$group`, `$sort`, and `$project`, then converts the result into a pandas DataFrame.

## Analytical Questions

In [16]:
insights = run_all_questions(combined_df)
save_keyword_chart(insights["keyword_summary"], ANALYTICS_DIR / "question1_top_keywords.png")
save_source_share_chart(insights["source_summary"], ANALYTICS_DIR / "question2_top_sources.png")
save_yearly_volume_chart(insights["yearly_summary"], ANALYTICS_DIR / "question3_yearly_volume.png")
save_language_distribution_chart(insights["language_summary"], ANALYTICS_DIR / "question4_language_distribution.png")
save_insight_summary(insights, ANALYTICS_DIR / "insight_summary.txt")
save_question_report(insights, ANALYTICS_DIR / "analytical_questions.txt")
insights.keys()

INFO: Insight | top_keyword=general_apple | mention_count=64
INFO: Insight | top_source=newsapi_Apple_page_1.json | share_pct=38.61
INFO: Insight | peak_year=2026 | mention_count=101
INFO: Insight | top_language=unknown | share_pct=86.14
INFO: Saved keyword insight chart | path=c:\Users\topic\OneDrive\Desktop\PROJECTS\Projekat za unstructured data\social_media_brand_monitor\data\processed\analytics\question1_top_keywords.png
INFO: Saved source-share chart | path=c:\Users\topic\OneDrive\Desktop\PROJECTS\Projekat za unstructured data\social_media_brand_monitor\data\processed\analytics\question2_top_sources.png
INFO: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before plotting.
INFO: Using categorical units to plot a list of strings that are all parsable as floats or dates. If these strings should be plotted as numbers, cast to the appropriate data type before pl

dict_keys(['keyword_summary', 'source_summary', 'yearly_summary', 'language_summary'])

### Question 1. Which Apple topic appears most often in the cleaned dataset?

In [17]:
question1 = top_keywords_by_mention_count(combined_df)
question1.head(10)

,primary_keyword,mention_count,avg_overview_chars,avg_rating
0,general_apple,64,153.34375,3.0
1,iphone,15,228.066667,3.0
2,apple services,8,218.5,3.0
3,airpods,4,235.0,3.0
4,tim cook,3,236.666667,3.0
5,macbook,3,258.0,3.0
6,apple watch,2,130.0,3.0
7,ipad,2,231.0,3.0


Quantified finding: the first row of `question1` identifies the top Apple topic and its exact mention count. Chart saved as `data/processed/analytics/question1_top_keywords.png`.

### Question 2. Which source contributes the largest share of Apple coverage?

In [18]:
question2 = source_share_summary(combined_df)
question2.head(10)

,source,mention_count,share_pct
0,newsapi_Apple_page_1.json,39,38.61
1,newsapi_Apple_page_3.json,25,24.75
2,newsapi_Apple_page_2.json,19,18.81
3,apple_ratings_large.csv,14,13.86
4,sample.csv,3,2.97
5,sample.xml,1,0.99


Quantified finding: the first row of `question2` identifies the top source, raw mention count, and percentage share. Chart saved as `data/processed/analytics/question2_top_sources.png`.

### Question 3. In which year does Apple mention volume peak?

In [19]:
question3 = yearly_release_volume(combined_df)
question3

,mention_year,mention_count,unique_sources
0,2026,101,6


Quantified finding: the maximum `mention_count` in `question3` identifies the peak year and its volume. Chart saved as `data/processed/analytics/question3_yearly_volume.png`.

### Question 4. What is the dominant language in Apple coverage?

In [20]:
question4 = language_distribution(combined_df)
question4

,language,mention_count,share_pct
0,unknown,87,86.14
1,en,7,6.93
2,de,7,6.93


Quantified finding: the first row of `question4` identifies the dominant language, its row count, and its percentage share. Chart saved as `data/processed/analytics/question4_language_distribution.png`.

## Pipeline Integration

The same analytics workflow is integrated into `src/pipeline/run_pipeline.py` through `run_analytics_pipeline(cleaned_csv_path)`, so the full project runs with one command:

```bash
python src/pipeline/run_pipeline.py
```